# Synthetic valley base model

This notebook loads the coastal synthetic-valley base model, maps its hydraulic conductivity, runs it, and compares the simulated heads at the end of the steady-state stress period to the true heads at the observation wells. The companion notebook [`synthetic-valley-b-advanced`](synthetic-valley-b-advanced.ipynb) replaces the simple boundary packages with the MODFLOW 6 advanced packages.

> **Model reference.** The synthetic valley model is described in Hughes, J.D., Langevin, C.D., Paulinski, S.R., Larsen, J.D., and Brakenhoff, D., 2024, FloPy workflows for creating structured and unstructured MODFLOW models: Groundwater, v. 62, no. 1, p. 124–139, https://doi.org/10.1111/gwat.13327.

## What this notebook covers

By the end of this notebook you will be able to:

- load an existing MODFLOW 6 simulation with `flopy.mf6.MFSimulation.load()` and read its arrays,
- map a property layer by layer with `flopy.plot.PlotMapView()`,
- run the model and plot the water table in plan view and in cross section with `flopy.plot.PlotCrossSection()`, and
- compare simulated heads to observed heads.

## The model

The synthetic valley drains south to the sea. The model uses a structured grid of 40 rows and 25 columns of 152.4 meter (m) cells, and five layers:

- layers 1 and 2 are the **upper aquifer**,
- layer 3 is a **clay** confining unit, present only in part of the valley, and
- layers 4 and 5 are the **lower aquifer**.

Where there is no clay, the layer 3 cells are **vertical pass-through cells** (`idomain` of -1). They have no volume, so water moves straight from layer 2 to layer 4.

The stresses are simple packages. Recharge (**RCH**) is applied at the land surface, a lake is represented by layer 1 cells with a very high hydraulic conductivity and evapotranspiration (**EVT**) from the lake surface, a stream is represented by River (**RIV**) cells, and the sea is a General-Head Boundary (**GHB**) at a head of 0 m along the southern edge. Two pumping wells (**WEL**) start in the second stress period.

The first stress period is a 16,071-day **steady-state** period, which gives the flow system before pumping. Twenty one-year transient periods follow. MODFLOW has no built-in units, so every input shares one length and one time unit: here meters and days.

## Imports

Import FloPy, Matplotlib, and NumPy. `sv_notebook_helpers` holds the location of the model files and two functions that read the true and simulated heads at the observation wells.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from sv_notebook_helpers import BASE_MODEL, simulated_heads, true_heads

## Load the model

Load the simulation with `flopy.mf6.MFSimulation.load()`, then point it at a new workspace with `set_sim_path()` so the original files in `examples/data` are left untouched. `pathlib.Path` builds the path to the new workspace.

In [ ]:
name = "sv"
new_ws = pl.Path("models/synthetic-valley-base")

sim = flopy.mf6.MFSimulation.load(
    sim_name=name,
    sim_ws=BASE_MODEL,
    write_headers=False,
    verbosity_level=0,
)
sim.set_sim_path(new_ws)
gwf = sim.get_model(name)
gwf.get_package_list()

Read the arrays the figures need from the packages: `idomain` from the discretization (**DIS**) package and the horizontal hydraulic conductivity `k` from the Node Property Flow (**NPF**) package. Each is a NumPy array of shape `(nlay, nrow, ncol)`, and FloPy counts layers, rows, and columns from 0, so `k11[0]` is layer 1.

The lake cells are the layer 1 cells with a hydraulic conductivity above 1,000 meters per day (m/d). The cross sections below run through the row and column at the center of the lake.

In [ ]:
idomain = gwf.dis.idomain.array
k11 = gwf.npf.k.array
extent = gwf.modelgrid.extent

lake = k11[0] > 1000.0
lake_rows, lake_cols = np.nonzero(lake)
row_xs = int(round(lake_rows.mean()))
col_xs = int(round(lake_cols.mean()))
print(
    f"{lake.sum()} lake cells; sections along row {row_xs + 1} and column {col_xs + 1}"
)

## Hydraulic conductivity

Map the horizontal hydraulic conductivity (**K**) of the upper aquifer, the clay, and the lower aquifer. Each panel is drawn by `flopy.plot.PlotMapView()` for one `layer`, and `plot_array()` colors the cells on a logarithmic scale (`mpl.colors.LogNorm`) because K spans several orders of magnitude. The lake cells are masked in layer 1 and drawn in blue, and the pass-through cells are masked in layer 3. `plot_bc()` marks the river (cyan) and coastal (purple) cells.

In [ ]:
width = 17.15 / 2.54  # figure width, inches
panels = {
    "A": (0, "Upper aquifer (layer 1)"),
    "B": (2, "Clay (layer 3)"),
    "C": (3, "Lower aquifer (layer 4)"),
}
lake_mask = np.ma.masked_where(~lake, np.ones(lake.shape))

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [list(panels)],
        figsize=(width, 0.62 * width),
        layout="constrained",
    )
    for letter, (k, heading) in panels.items():
        ax = axd[letter]
        mask = idomain[k] < 1
        if k == 0:
            mask = mask | lake
        kk = np.ma.masked_where(mask, k11[k])
        norm = mpl.colors.LogNorm(vmin=kk.min(), vmax=kk.max())

        mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=k)
        pa = mm.plot_array(kk, norm=norm)
        if k == 0:
            mm.plot_array(lake_mask, cmap="Blues", vmin=0.0, vmax=1.5)
            mm.plot_bc(package=gwf.riv, color="cyan")
        mm.plot_bc(package=gwf.ghb, color="purple", kper=0)
        ax.set_aspect("equal")
        ax.set_xlabel("x (m)")
        if letter == "A":
            ax.set_ylabel("y (m)")
        flopy.plot.styles.heading(ax=ax, letter=letter, heading=heading)

        # three ticks spread evenly on the log scale across the layer's range
        cb = fig.colorbar(pa, ax=ax, orientation="horizontal", shrink=0.9)
        cb.set_ticks(np.geomspace(kk.min(), kk.max(), 3))
        cb.ax.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter("%.2g"))
        cb.ax.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
        cb.set_label("K (m/d)")

In [ ]:
for letter, (k, heading) in panels.items():
    active = idomain[k] > 0
    if k == 0:
        active = active & ~lake
    print(
        f"{heading}: K from {k11[k][active].min():.4g} to {k11[k][active].max():.4g} m/d"
    )
print(f"clay cells: {(idomain[2] > 0).sum()} of {idomain[2].size}")

**What to look for.** The upper aquifer (A) and the lower aquifer (C) have the same K, from 16.1 to 133.3 m/d, highest in the north of the valley. The clay (B) is present in 346 of the 1,000 cells, mostly in the northern half under the lake, and its K of 0.0279 to 0.0432 m/d is at least 370 times lower than any aquifer cell. Where the clay is present it slows the water moving between the upper and lower aquifers. Where it is absent, the pass-through cells connect the two aquifers directly.

## Run the model

Write the input files to the new workspace with `write_simulation()` and run MODFLOW 6 with `run_simulation()`, which returns `True` when the run ends normally.

In [ ]:
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

## Water table at the end of the steady-state period

Read the heads at the end of the steady-state period with `gwf.output.head().get_data()`. `kstpkper=(0, 0)` selects the first time step of the first stress period, again counting from 0. Inactive cells hold a head of 1.0E+30, which `masked_values` leaves out of the plots.

Map the head in layer 1, which is the water table, and draw it along a west-east and a north-south cross section through the center of the lake. In the cross sections, `plot_array(head, head=head)` colors each cell only up to the simulated head, so the colored area ends at the water table and the unsaturated part of layer 1 above it is left white. `plot_ibound()` shades the pass-through cells in layer 3 gray. The sections show the top 25 m of the model, from the land surface down through the clay, because the upper aquifer is only a few meters thick and the water table would be lost in a section drawn to the bottom of the model.

In [ ]:
head = gwf.output.head().get_data(kstpkper=(0, 0))
water_table = np.ma.masked_greater(head[0], 1.0e29)
vmin, vmax = np.floor(water_table.min()), np.ceil(water_table.max())
levels = np.arange(vmin, vmax + 0.5, 0.5)

top = gwf.dis.top.array
x_ns = gwf.modelgrid.xcellcenters[0, col_xs]
y_ew = gwf.modelgrid.ycellcenters[row_xs, 0]
# elevation range of the sections: land surface to 25 m below its highest point
ylim = (np.ceil(top.max()) - 25.0, np.ceil(top.max()))
lake_3d = np.zeros(idomain.shape, dtype=bool)
lake_3d[0] = lake
sections = {
    "B": ({"row": row_xs}, "West-east section", "Distance from the west edge (m)"),
    "C": (
        {"column": col_xs},
        "North-south section",
        "Distance from the north edge (m)",
    ),
}

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"], ["A", "C"]],
        figsize=(width, 0.85 * width),
        layout="constrained",
        width_ratios=[1.0, 1.5],
    )
    ax = axd["A"]
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    pa = mm.plot_array(head, masked_values=[1.0e30], vmin=vmin, vmax=vmax)
    cs = mm.contour_array(
        head, masked_values=[1.0e30], levels=levels, colors="black", linewidths=0.5
    )
    ax.clabel(cs, fmt="%g", fontsize=6)
    mm.plot_array(lake_mask, cmap="Blues", vmin=0.0, vmax=1.5, alpha=0.6)
    mm.plot_bc(package=gwf.riv, color="cyan")
    mm.plot_bc(package=gwf.ghb, color="purple", kper=0)
    ax.plot(extent[:2], [y_ew, y_ew], color="red", lw=1.0, ls="--")
    ax.plot([x_ns, x_ns], extent[2:], color="red", lw=1.0, ls="--")
    ax.text(extent[0], y_ew, "B ", color="red", ha="right", va="center")
    ax.text(x_ns, extent[3], "C", color="red", ha="center", va="bottom")
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    flopy.plot.styles.heading(ax=ax, letter="A", heading="Water table")
    fig.colorbar(pa, ax=ax, orientation="horizontal", shrink=0.9, label="Head (m)")

    for letter, (line, heading, xlabel) in sections.items():
        ax = axd[letter]
        xs = flopy.plot.PlotCrossSection(model=gwf, ax=ax, line=line)
        xs.plot_array(head, head=head, masked_values=[1.0e30], vmin=vmin, vmax=vmax)
        xs.plot_array(
            np.ma.masked_where(~lake_3d, np.ones(idomain.shape)),
            head=head,
            cmap="Blues",
            vmin=0.0,
            vmax=1.5,
            alpha=0.6,
        )
        xs.plot_ibound(color_vpt="0.85")
        xs.plot_grid(lw=0.25, color="0.5")
        xs.plot_surface(top, color="saddlebrown", lw=1.0)
        xs.plot_surface(water_table, color="blue", lw=1.0)
        ax.set_ylim(ylim)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Elevation (m)")
        flopy.plot.styles.heading(ax=ax, letter=letter, heading=heading)
    handles = [
        mpl.lines.Line2D([], [], color="saddlebrown", lw=1.0, label="Land surface"),
        mpl.lines.Line2D([], [], color="blue", lw=1.0, label="Water table"),
    ]
    flopy.plot.styles.graph_legend(
        ax=axd["C"],
        handles=handles,
        labels=[h.get_label() for h in handles],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.3),
        ncol=2,
    )

In [ ]:
print(f"water table from {water_table.min():.2f} to {water_table.max():.2f} m")
print(
    f"lake cells from {water_table[lake].min():.2f} to {water_table[lake].max():.2f} m"
)

**What to look for.** The water table (A) falls from 4.40 m in the north to 0.03 m at the coast. The head is 3.98 m in every lake cell, because their very high K makes the lake surface flat. South of the lake the contours bend upstream along the river, which drains the upper aquifer. In the sections (B and C) the water table (blue) lies within a few meters of the land surface (brown), and the gray cells in layer 3 are where the clay is absent.

## Compare the simulated heads to the true heads

The observation wells measure the water table (layer 1) and the lower aquifer (layers 4 and 5) at 17 locations. The true heads come from the Voronoi-grid advanced model in `examples/data/synthetic-valley/data`, which defines the truth for the class. `true_heads()` returns them and `simulated_heads()` reads the simulated heads the model saved at the same wells, each as a table with one column per well and one row per output time. Select the end of the steady-state period with `.loc[]`.

A **residual** is the simulated head minus the true head. A positive residual means the model is too high.

In [ ]:
t_steady = gwf.output.head().get_times()[0]
kinds = {"A": ("wt", "Water table"), "B": ("aq", "Lower aquifer")}

residuals = {}
for letter, (kind, label) in kinds.items():
    true = true_heads(kind).loc[t_steady]
    sim_head = simulated_heads(new_ws, kind).loc[t_steady, true.index]
    residuals[letter] = (true, sim_head, sim_head - true)
    r = sim_head - true
    rmse = np.sqrt((r**2).mean())
    print(
        f"{label}: true heads {true.min():.3f} to {true.max():.3f} m; "
        f"mean residual {r.mean():.3f} m, root-mean-square {rmse:.3f} m, "
        f"largest {r.abs().max():.3f} m"
    )

Plot the simulated heads with the true heads on a one-to-one plot. A well whose simulated head equals its true head plots on the diagonal line.

In [ ]:
values = np.concatenate([np.r_[t, s] for t, s, _ in residuals.values()])
lim = (np.floor(values.min()), np.ceil(values.max()))

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [list(kinds)], figsize=(width, 0.5 * width), layout="constrained"
    )
    for letter, (kind, label) in kinds.items():
        true, sim_head, _ = residuals[letter]
        ax = axd[letter]
        ax.plot(lim, lim, color="0.5", lw=0.75, label="One-to-one line")
        ax.plot(true, sim_head, "o", ms=4, mfc="none", color="black", label="Well")
        ax.set_xlim(lim)
        ax.set_ylim(lim)
        ax.set_aspect("equal")
        flopy.plot.styles.heading(ax=ax, letter=letter, heading=label)
        flopy.plot.styles.xlabel(ax=ax, label="True head (m)")
        flopy.plot.styles.ylabel(ax=ax, label="Simulated head (m)")
    flopy.plot.styles.graph_legend(ax=axd["A"], loc="upper left")

**What to look for.** The true heads range from 0.53 to 4.10 m at the water table and from 0.53 to 3.95 m in the lower aquifer. The simulated heads are 0.06 m low on average at the water table (A) and 0.09 m low in the lower aquifer (B). The root-mean-square residuals are 0.23 and 0.20 m, and no well is off by more than 0.59 m. The base model represents the lake, stream, and recharge with simple packages and uses one K field for both aquifers, so it cannot reproduce the truth exactly. The companion notebook replaces those packages with the advanced packages.

## Recap

You loaded the coastal synthetic-valley base model with FloPy, mapped the hydraulic conductivity of the upper aquifer, the clay, and the lower aquifer, ran the model, drew the steady-state water table in plan view and along two cross sections through the lake, and compared the simulated heads at 17 observation wells to the true heads.